# Iranian Customer Churn Prediction

Notebook ini membangun baseline customer churn prediction dari dataset Iranian Churn. Struktur dibuat bertahap agar mudah dipindahkan ke file `.py` dan diintegrasikan ke n8n.

## 1. Import Library

In [ ]:
from pathlib import Path
import json
import warnings
import importlib.util

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = Path("Customer Churn.csv")
ARTIFACT_DIR = Path("artifacts")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", palette="Set2")

## 2. Import Data

In [ ]:
def clean_column_names(columns):
    return (
        pd.Index(columns)
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        .str.strip("_")
    )


df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df.columns = clean_column_names(df.columns)

display(df.head())
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

In [ ]:
target_col = "churn"
raw_feature_columns = [column for column in df.columns if column != target_col]

print("Columns:")
for column in df.columns:
    print(f"- {column}")

## 3. EDA

In [ ]:
data_quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique_count": df.nunique(),
})

display(data_quality)
print(f"Duplicated rows: {df.duplicated().sum():,}")

In [ ]:
display(df.describe().T)

In [ ]:
target_summary = (
    df[target_col]
    .value_counts()
    .rename_axis(target_col)
    .reset_index(name="count")
)
target_summary["percentage"] = target_summary["count"] / len(df) * 100

display(target_summary)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Churn Class Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()

plt.figure(figsize=(12, 9))
sns.heatmap(
    df[numeric_columns].corr(),
    cmap="vlag",
    center=0,
    linewidths=0.4,
    annot=False,
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
selected_numeric = [
    "call_failure",
    "subscription_length",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "customer_value",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for axis, column in zip(axes, selected_numeric):
    sns.boxplot(data=df, x=target_col, y=column, ax=axis)
    axis.set_title(column.replace("_", " ").title())

plt.tight_layout()
plt.show()

In [ ]:
categorical_candidates = ["complains", "age_group", "tariff_plan", "status"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for axis, column in zip(axes, categorical_candidates):
    churn_rate = df.groupby(column)[target_col].mean().reset_index()
    sns.barplot(data=churn_rate, x=column, y=target_col, ax=axis)
    axis.set_title(f"Churn Rate by {column.replace('_', ' ').title()}")
    axis.set_ylabel("Churn Rate")

plt.tight_layout()
plt.show()

In [ ]:
churn_profile = df.groupby(target_col)[raw_feature_columns].mean().T
churn_profile.columns = ["not_churn", "churn"]
churn_profile["difference"] = churn_profile["churn"] - churn_profile["not_churn"]
churn_profile["abs_difference"] = churn_profile["difference"].abs()

churn_profile.sort_values("abs_difference", ascending=False).head(10)

## 4. Feature Engineering

In [ ]:
df_model = df.drop_duplicates().reset_index(drop=True)

print(f"Rows before duplicate handling: {len(df):,}")
print(f"Rows after duplicate handling: {len(df_model):,}")

In [ ]:
def add_engineered_features(data):
    data = data.copy()
    data["usage_minutes"] = data["seconds_of_use"] / 60
    data["failed_call_rate"] = data["call_failure"] / (data["frequency_of_use"] + 1)
    data["sms_share"] = data["frequency_of_sms"] / (
        data["frequency_of_sms"] + data["frequency_of_use"] + 1
    )
    data["value_per_usage_minute"] = data["customer_value"] / (data["usage_minutes"] + 1)
    data["value_per_frequency"] = data["customer_value"] / (data["frequency_of_use"] + 1)
    data["calls_per_distinct_number"] = data["frequency_of_use"] / (
        data["distinct_called_numbers"] + 1
    )
    return data


X_raw = df_model.drop(columns=target_col)
y = df_model[target_col]
X = add_engineered_features(X_raw)

categorical_features = ["complains", "age_group", "tariff_plan", "status"]
numeric_features = [column for column in X.columns if column not in categorical_features]

print(f"Feature count after engineering: {X.shape[1]}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
display(X.head())

## 5. Model Development

In [ ]:
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.25,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "rows": [len(X_train), len(X_valid), len(X_test)],
    "churn_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
})

display(split_summary)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ],
    verbose_feature_names_out=False,
)

model_candidates = {
    "logistic_regression": LogisticRegression(
        max_iter=2_000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

In [ ]:
def evaluate_classifier(model, features, target, threshold=0.5):
    probability = model.predict_proba(features)[:, 1]
    prediction = (probability >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(target, prediction),
        "precision": precision_score(target, prediction, zero_division=0),
        "recall": recall_score(target, prediction, zero_division=0),
        "f1": f1_score(target, prediction, zero_division=0),
        "roc_auc": roc_auc_score(target, probability),
        "pr_auc": average_precision_score(target, probability),
    }


def find_best_threshold(target, probability):
    precision, recall, thresholds = precision_recall_curve(target, probability)
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_index = int(np.argmax(f1_values))
    return float(thresholds[best_index]), float(f1_values[best_index])

In [ ]:
trained_models = {}
validation_rows = []

for model_name, estimator in model_candidates.items():
    pipeline = Pipeline(steps=[
        ("preprocess", clone(preprocess)),
        ("model", clone(estimator)),
    ])
    pipeline.fit(X_train, y_train)
    trained_models[model_name] = pipeline

    metrics = evaluate_classifier(pipeline, X_valid, y_valid)
    metrics["model"] = model_name
    validation_rows.append(metrics)

validation_results = (
    pd.DataFrame(validation_rows)
    .set_index("model")
    .sort_values(["pr_auc", "f1"], ascending=False)
)

display(validation_results)

In [ ]:
best_model_name = validation_results.index[0]
best_validation_model = trained_models[best_model_name]
valid_probability = best_validation_model.predict_proba(X_valid)[:, 1]
best_threshold, best_valid_f1 = find_best_threshold(y_valid, valid_probability)

print(f"Best model: {best_model_name}")
print(f"Best validation threshold: {best_threshold:.4f}")
print(f"Best validation F1 at threshold: {best_valid_f1:.4f}")

In [ ]:
final_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocess)),
    ("model", clone(model_candidates[best_model_name])),
])
final_pipeline.fit(X_train_valid, y_train_valid)

test_metrics = evaluate_classifier(final_pipeline, X_test, y_test, threshold=best_threshold)
display(pd.DataFrame([test_metrics], index=[best_model_name]))

In [ ]:
test_probability = final_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= best_threshold).astype(int)

print(classification_report(y_test, test_prediction, target_names=["not_churn", "churn"]))

confusion = confusion_matrix(y_test, test_prediction)
plt.figure(figsize=(5, 4))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["not_churn", "churn"],
    yticklabels=["not_churn", "churn"],
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
def to_python_value(value):
    if isinstance(value, (np.integer, np.int64)):
        return int(value)
    if isinstance(value, (np.floating, np.float64)):
        return float(value)
    return value


model_feature_columns = X.columns.tolist()
metadata = {
    "dataset": DATA_PATH.name,
    "target": target_col,
    "raw_feature_columns": raw_feature_columns,
    "model_feature_columns": model_feature_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "best_model": best_model_name,
    "threshold": best_threshold,
    "test_metrics": {key: to_python_value(value) for key, value in test_metrics.items()},
}

ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(final_pipeline, ARTIFACT_DIR / "iranian_churn_model.joblib")
with open(ARTIFACT_DIR / "model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(f"Saved model to: {ARTIFACT_DIR / 'iranian_churn_model.joblib'}")
print(f"Saved metadata to: {ARTIFACT_DIR / 'model_metadata.json'}")

In [ ]:
def prepare_model_input(data):
    if isinstance(data, dict):
        customer_data = pd.DataFrame([data])
    else:
        customer_data = pd.DataFrame(data).copy()

    customer_data.columns = clean_column_names(customer_data.columns)
    customer_data = add_engineered_features(customer_data)
    return customer_data[model_feature_columns]


def predict_churn(data, model=final_pipeline, threshold=best_threshold):
    prepared_data = prepare_model_input(data)
    probability = model.predict_proba(prepared_data)[:, 1]
    prediction = (probability >= threshold).astype(int)
    return pd.DataFrame({
        "churn_probability": probability,
        "churn_prediction": prediction,
    })


sample_payload = X_raw.iloc[0].to_dict()
display(predict_churn(sample_payload))

## 6. Explainable AI

In [ ]:
permutation = permutation_importance(
    final_pipeline,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(permutation_importance_df.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(
    data=permutation_importance_df.head(15),
    x="importance_mean",
    y="feature",
)
plt.title("Top Permutation Importances")
plt.xlabel("Average Precision Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
def get_transformed_features(model, features):
    transformer = model.named_steps["preprocess"]
    transformed = transformer.transform(features)
    feature_names = transformer.get_feature_names_out()
    return pd.DataFrame(transformed, columns=feature_names, index=features.index)


def select_positive_class_shap_values(shap_values):
    if isinstance(shap_values, list):
        return shap_values[1]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        return shap_values[:, :, 1]
    return shap_values


if importlib.util.find_spec("shap") is None:
    print("SHAP is not installed. Permutation importance above is available as model-agnostic XAI.")
else:
    import shap

    X_train_sample = X_train_valid.sample(min(250, len(X_train_valid)), random_state=RANDOM_STATE)
    X_test_sample = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
    X_train_transformed = get_transformed_features(final_pipeline, X_train_sample)
    X_test_transformed = get_transformed_features(final_pipeline, X_test_sample)
    fitted_estimator = final_pipeline.named_steps["model"]

    try:
        if best_model_name in {"random_forest", "gradient_boosting"}:
            explainer = shap.TreeExplainer(fitted_estimator)
            shap_values = explainer.shap_values(X_test_transformed)
        elif best_model_name == "logistic_regression":
            explainer = shap.LinearExplainer(fitted_estimator, X_train_transformed)
            shap_values = explainer.shap_values(X_test_transformed)
        else:
            explainer = shap.Explainer(fitted_estimator.predict_proba, X_train_transformed)
            shap_values = explainer(X_test_transformed).values

        shap_values_positive = select_positive_class_shap_values(shap_values)
        shap.summary_plot(shap_values_positive, X_test_transformed, show=False, max_display=15)
        plt.title("SHAP Summary Plot")
        plt.tight_layout()
        plt.show()
    except Exception as error:
        print(f"SHAP explanation skipped: {error}")

In [ ]:
prediction_audit = X_test.copy()
prediction_audit["actual_churn"] = y_test
prediction_audit["predicted_probability"] = test_probability
prediction_audit["predicted_churn"] = test_prediction

prediction_audit.sort_values("predicted_probability", ascending=False).head(10)